# SODA: from instrument files to one link you can share

This notebook turns the files your instrument (or analysis pipeline) wrote into the static
files SODA reads in the browser. You put each dataset in its own folder (Section 1), the
notebook works out what each one is, converts it and writes **one folder, `export/`**, that you
upload to the cloud as it is. At the end it gives you **one link** that opens all your datasets.

You fill in only two things, in Section 0: **where to work** (`WORK`) and **your cloud folder's
link** (`CLOUD_URL`). Everything else has a default. Before the upload, Section 7 asks one
question: **is this data public?**

| Section | What happens | Output |
|---|---|---|
| 0. Setup | finds the repository, checks Python packages, asks for your work folder and cloud link | |
| 1. Put your files here | one folder per dataset, any name | `inputs/` |
| 2. Check the inputs | says what each folder is, how sure that is and anything missing | a table per dataset |
| 3. Prepare | reads each dataset once into a working copy | `cache/` (never upload) |
| 4. Export | cuts images into tiles, writes gene files and UMAPs, lists the datasets | `export/`, `datasets.json` |
| 5. De-identification check | scans `export/` for sample IDs (when you have a key file) | must say `clean` |
| 6. Look before you upload | size per dataset, then open it on your own computer | a local test link |
| 7. Upload and share | asks "is this data public?", prints the upload command and the link, then tests the link | **the link to share** |

Run the cells from top to bottom. Each step skips datasets that are already done, so you can
stop and start again. Commands are the same ones in `docs/PREPARING_DATA.md`.

## 0. Setup

Use a separate Python environment for SODA, because it needs older versions of `zarr` and
`tifffile` than many other tools. Once, from the repository folder (keep the path short on Windows),
with Python 3.12 or 3.13 installed:

```
powershell -ExecutionPolicy Bypass -File install.ps1    (macOS / Linux: bash install.sh)
.venv\Scripts\activate              (macOS / Linux: source .venv/bin/activate)
pip install jupyter
```

The install script makes `.venv`, puts the exact tested package versions in it
(`web/SpatialViewer/requirements-lock.txt`) and ends with **SODA is installed**; it is safe to run again.
Then open this notebook with that environment's Jupyter (or pick `.venv` as the kernel in VS Code).
By hand instead: `pip install -r web/SpatialViewer/requirements.txt -r web/SpatialViewer/requirements-clustering.txt`
in your own venv (the second file is optional: without it the export skips the whole-section UMAP and clusters).

In [ ]:
import os, sys, json, shutil, subprocess, time, importlib
from pathlib import Path

def find_repo(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "web" / "SpatialViewer" / "prepare.py").exists():
            return p
    raise FileNotFoundError("Open this notebook from inside the SODA repository "
                            "(the folder that has web/SpatialViewer/prepare.py), or set REPO by hand.")

REPO = find_repo()
SV = REPO / "web" / "SpatialViewer"          # the scripts: prepare.py, deid.py, static_serve.py
PY = sys.executable                          # the same Python this notebook runs on
print("repository:", REPO)
print("python:    ", PY, sys.version.split()[0])

need = {"numpy": "core", "tifffile": "core", "zarr": "core", "imagecodecs": "core", "PIL": "core",
        "h5py": "core", "pyarrow": "core", "scipy": "core", "cryptography": "core",
        "sklearn": "UMAP (optional)", "umap": "UMAP (optional)", "igraph": "UMAP (optional)",
        "leidenalg": "UMAP (optional)"}
missing = []
for mod, what in need.items():
    try:
        m = importlib.import_module(mod)
        print(f"  ok       {mod:12s} {getattr(m, '__version__', '')}")
    except ImportError:
        print(f"  MISSING  {mod:12s} ({what})")
        missing.append(mod)
def _major(mod):
    try:
        return int(importlib.import_module(mod).__version__.split(".")[0])
    except Exception:
        return None
wrong = []
if (_major("zarr") or 0) >= 3:
    wrong.append("zarr must be 2.x (found 3+): images cannot be written")
if (_major("tifffile") or 0) >= 2025:
    wrong.append("tifffile must be older than 2025 (it needs zarr 3)")
for w in wrong:
    print("  WRONG VERSION:", w)
if wrong or any(need[m] == "core" for m in missing):
    print("\nThis Python is not set up for SODA. Make the .venv above, install the requirements in it,"
          "\nand run this notebook with that environment. Stop here until this cell shows no MISSING core or WRONG lines.")

### Settings: edit this cell

You fill in only these two:

- **`WORK`** is one folder on a big, fast disk. Everything goes inside it:
  `inputs/` (your files), `cache/` (working copies, large, never uploaded) and
  `export/` (what you upload). Plan for about 2-3 times the size of your raw files.
- **`CLOUD_URL`** is the link to the folder in the cloud where the data will go
  (on Azure: a blob container, e.g. `https://myaccount.blob.core.windows.net/spatial`).
  Paste only the address, never a key or a `?sv=...` pass. You can leave it empty for now
  and fill it in before Section 7.

In [ ]:
WORK = Path(r"D:\SODA_work")   # EDIT: a big disk; inputs/, cache/ and export/ go inside
CLOUD_URL = ""                 # EDIT before Section 7: your cloud folder's link, e.g. https://myaccount.blob.core.windows.net/spatial

print("work folder: ", WORK)
print("cloud folder:", CLOUD_URL.strip().split("?", 1)[0] or "(not set yet: paste it before Section 7)")

### Advanced settings (optional)

**You do not need to change anything here.** Run the cell anyway: it sets the defaults that
the rest of the notebook uses.

- **`KEY_FILE`** is the de-identification key: it maps the numbers used in the export back to
  your real sample IDs. Keep it in a private folder **outside** `WORK` and outside the repository,
  and never upload or share it. See `docs/DEIDENTIFICATION.md`. The sample ID check in Section 5
  runs only when this file exists (or when `ALWAYS_CHECK_IDS = True`).
- **`SODA_PAGE`** is the address of the SODA page people open with your link: the shared SODA
  page, or your own copy of SODA. Section 7 builds the link from it.
- **`TITLE`** names the whole collection at the top of the page.
- **`READ_PASS`** is only for a private cloud folder: a read-only pass (on Azure a SAS, starting
  with `sv=`). It goes into the link, so **anyone with the link can read the data** until the
  pass expires.
- The rest changes how images and genes are exported.

In [ ]:
# Sample ID check (Section 5)
KEY_FILE = Path(os.environ.get("SPATIALVIZ_KEY") or Path.home() / "private" / "soda" / "sample_id_key.csv")  # your de-identification key, if you have one
ALWAYS_CHECK_IDS = False   # True: run the sample ID check even without a key file

# The link you share (Section 7)
SODA_PAGE = "<the SODA page's address>"  # e.g. "https://soda.example.org/". The shared SODA page's address
                                         # is not decided yet: put it (or your own copy's address) here
TITLE = ""                 # a name for the whole collection, e.g. "Our nerve atlas" ("" keeps the one already set)
READ_PASS = ""             # private folder only: a READ-ONLY pass (Azure SAS, starts with sv=). Anyone with
                           # the link can then read the data until the pass expires. Never a write pass.

# Export
TILE_FORMAT = "jpg"        # "jpg" is several times smaller; "png" is lossless (use it when exact intensities matter)
VISIUM_BINS = "008um,016um"  # Visium HD layers to keep. The 2 um layer is huge and mostly one transcript per bin
WORKERS = 8                # parallel tile writers during export
MARKERS = {                # marker-gene lists for suggested cluster names ("" = the generic list)
    "visium": "",          # e.g. str(SV / "markers" / "nerve.json") or "brain.json"
    "proteomics": "",      # e.g. str(SV / "markers" / "akoya_nerve.json")
}
REBUILD = False            # True: prepare datasets again even when cache/<id>/ already has them (Section 3)

INPUTS, CACHE, EXPORT = WORK / "inputs", WORK / "cache", WORK / "export"
for d in (INPUTS, CACHE, EXPORT):
    d.mkdir(parents=True, exist_ok=True)
os.environ["SPATIALVIZ_KEY"] = str(KEY_FILE)  # every script started from here uses this key
print("key file:   ", KEY_FILE, "(found: Section 5 checks for sample IDs)" if KEY_FILE.exists()
      else "(none: Section 5 skips the sample ID check)")
if KEY_FILE.resolve().is_relative_to(WORK.resolve()) or KEY_FILE.resolve().is_relative_to(REPO.resolve()):
    print("WARNING: keep the key file outside the work folder and the repository.")
print("SODA page:  ", SODA_PAGE if SODA_PAGE.strip() and "<" not in SODA_PAGE else "(not set: Section 7 prints the link with a placeholder)")

## 1. Put your files here

**Put each dataset in its own folder under `inputs/`. Any folder name works, and you do not
have to say what kind of data it is:** Section 2 looks at the files and works it out. Then
run every cell.

```
inputs/
  nerve_section1/      <- for example a Space Ranger outs/ folder + the microscope image
  tumour_xenium/       <- a Xenium output folder, exactly as the instrument wrote it
  slide_07.qptiff      <- one image file (QPTIFF, OME-TIFF) or one .h5ad can sit here on its own
```

The folder name becomes the dataset id (letters, digits, `_` and `-`; anything else becomes
`_`), so do not put patient or sample identifiers in it. What SODA recognises:

| Data | What the folder holds |
|---|---|
| Visium HD / Visium | Space Ranger's `outs/` folder **and the full-resolution image you gave `spaceranger count --image`** next to it (not `tissue_hires_image.png`) |
| Xenium | the output folder: `morphology_focus/`, `experiment.xenium`, cells, transcripts (stains alone are enough); `cell_boundaries` and `nucleus_boundaries` (`.parquet` or `.csv.gz`) give the cell and nucleus outlines |
| CosMx SMI | the flat files `*_exprMat_file`, `*_metadata_file`, `*_fov_positions_file`, `*_tx_file` and the image folders (`CellLabels/`, or a `*-polygons.csv`, gives the cell outlines); one slide per folder |
| MERSCOPE | one region folder: `detected_transcripts`, `cell_by_gene` + `cell_metadata`, `images/`; `cell_boundaries.parquet` (or `cell_boundaries/*.hdf5`) gives the cell outlines |
| Stereo-seq | SAW output: `<SN>_HE_regist.tif` (the registered image) and `<SN>.tissue.gef` / `.raw.gef` / `.gem.gz`, `<SN>.cellbin.gef` (the cells, with their outlines) |
| GeoMx DSP | the `*.dcc` files, the `*.pkc` probe kit, the annotation sheet, and the scans as `*.ome.tiff` if you have them |
| Multiplexed imaging, H&E | one `.qptiff` / `.ome.tif` / `.tif` file, or a folder with one TIFF per marker (`NN_Marker_..._gray.tif`) |
| Single-cell RNA-seq | one `.h5ad` with the UMAP in `obsm["X_umap"]`, or `filtered_feature_bc_matrix/` with `analysis/` or a `umap.csv` |

The exact files of each instrument, and what each one becomes, are in `docs/PREPARING_DATA.md`.

- **Not sure, or it guessed wrong?** Section 2 shows what it found for every folder. When it is
  not certain it does not guess: it lists the choices and you pick one in `OPTIONS` there.
- **Zipped output** must be unzipped first (Section 2 tells you which zip holds what).
- **Big files you don't want to copy:** leave them where they are and point the dataset at them
  with `src` in Section 2's `OPTIONS`.
- **Your own UMAP and clusters (Visium):** put `my_umap.csv` (`barcode,UMAP_1,UMAP_2`),
  `my_clusters.csv` (`barcode,seurat_clusters,cell_type,...`) or `my_analysis.h5ad` next to
  `outs/` and set `user_layer` (e.g. `"008um"`) in `OPTIONS`; see `docs/PREPARING_DATA.md`,
  "Your own UMAP and clusters".
- The layout of earlier versions, `inputs/<kind>/<dataset_id>/` (`inputs/visium/...`,
  `inputs/xenium/...`, `inputs/proteomics_file/...`), still works: the kind folder's name is
  then taken as what you say the data is.

In [ ]:
readme = INPUTS / "README.txt"
if not readme.exists():
    readme.write_text("Put each dataset in its own folder here. Any name works; the name becomes the dataset id.\n"
                      "SODA works out what each folder holds: run Section 2 of notebooks/SODA_pipeline.ipynb.\n"
                      "What each instrument's folder should contain: docs/PREPARING_DATA.md.\n", encoding="utf-8")
here = sorted(p.name + ("/" if p.is_dir() else "") for p in INPUTS.iterdir() if p.name != "README.txt")
print(f"{INPUTS}:")
print("  " + ("\n  ".join(here) if here else "empty. Put one folder per dataset in it, then run Section 2."))

## 2. Check the inputs

The next cell looks at every folder under `inputs/` (names, sizes and file headers only; nothing
is converted yet) and shows one table per dataset: what it is, **how sure** that is, what was
found, and what is missing. It is the same check as `python web/SpatialViewer/prepare.py detect <folder>`.

- **certain**: one kind fits and all its required files are there. Nothing to do.
- **NOT SURE: pick one**: more than one kind fits, or a choice is open (for example two images
  next to a Visium `outs/`). The table prints the line to copy into `OPTIONS`. It never guesses.
- **recognised, but something is missing**: the table says which file.
- **not recognised**: say what it is with `"kind"`, or check the folder against Section 1.

**`OPTIONS` is the one place to change anything about a dataset.** Keyed by dataset id (the
folder's name). Leave a dataset out to use what was detected and the defaults.

| Key | Meaning |
|---|---|
| `kind` | you say what the folder is: `"visium"`, `"xenium"`, `"cosmx"`, `"merscope"`, `"stereoseq"`, `"geomx"`, `"tiff"` (one image file), `"akoya"` (one TIFF per marker) or `"rnaseq"`. Used as given, also when detection is unsure; the table still lists that kind's missing files |
| `image` | Visium: the full-resolution image given to `spaceranger count --image`, when it is not next to `outs/` or several TIFFs are. Stereo-seq: the registered image. GeoMx: a scan, or a list of scans |
| `gef`, `gem`, `cellbin`, `tissue_mask` | Stereo-seq: use this file instead of the one picked |
| `dcc`, `pkc`, `annotation` | GeoMx: use these instead of the ones found (`pkc` and `annotation` may be lists) |
| `umap`, `obs` | single-cell RNA-seq from a 10x matrix folder: the UMAP CSV, and a list of colour-by CSVs |
| `label` | the name the picker shows, e.g. `"Normal nerve"` (no sample IDs) |
| `group` | picker heading: `"proteomics"`, `"transcriptomics"` or `"scrnaseq"` (set automatically) |
| `src` | use files from another place instead of the `inputs/` subfolder (file or folder, same layout) |
| `pixel_size` | µm per full-resolution pixel, when the file does not carry it |
| `channels` | proteomics_file only: channel names in order, comma-separated |
| `crop` | `"x0,y0,x1,y1"` in full-resolution pixels: keep only that rectangle (not Xenium, CosMx or MERSCOPE) |
| `flip` | `"h"`, `"v"` or `"hv"`: mirror the image (not Xenium, CosMx or MERSCOPE) |
| `image_scale` | Visium only: your image width ÷ the width of the image Space Ranger used |
| `user_layer` | Visium only: the layer your own UMAP / clusters belong to, e.g. `"008um"`, `"016um"` or `"spots"` (needed when you give them) |
| `user_umap` | your embedding CSV (`barcode,UMAP_1,UMAP_2`); default `my_umap.csv` in the dataset folder |
| `user_obs` | your clusters CSV, or a list of them; default `my_clusters.csv` and `my_clusters_*.csv` in the dataset folder |
| `user_h5ad` | an `.h5ad` instead of the CSVs; default `my_analysis.h5ad` in the dataset folder |
| `user_embedding` | the `obsm` key of the embedding in `user_h5ad` (default `X_umap`) |
| `user_columns` | only these columns become clusterings, e.g. `"seurat_clusters,cell_type"` |
| `user_barcode_prefix` / `user_barcode_suffix` | a merged object with several sections: keep this section's barcodes (e.g. `"S1_"`, `"_2"`) |
| `user_replace` | `True`: show only your clusterings, not SODA's Leiden ones |
| `no_cells` | Xenium, CosMx, MERSCOPE: `True` skips the cells layer |
| `bins` | Stereo-seq only: bin sizes in DNBs, e.g. `"50,100"` (default `"20,50,100"` = 10, 25 and 50 µm) |
| `no_transcripts` | Xenium, CosMx, MERSCOPE: `True` skips the molecules (the slowest part of a Xenium prepare) |
| `qv_min` | Xenium only: lowest transcript quality value kept (default 20) |
| `images` | CosMx: `"morphology"`, `"composite"` or `"none"` instead of the automatic choice; MERSCOPE: `"none"` (a transcript-density background instead of the images) |
| `z` | MERSCOPE: the z-plane of the mosaic images to use (default: the middle one present); CosMx: the plane of raw z-stack images |
| `cells_dir` | MERSCOPE only: the folder holding `cell_by_gene.csv` and `cell_metadata.csv` when there are several segmentation runs |
| `layout` | GeoMx only: `"grid"` draws the ROIs on a labelled grid even when scan images are there |
| `keep_slide_names` | GeoMx only: `True` keeps the slide / scan names as group names (default: Slide 1, Slide 2 ...) |
| `sheet` | GeoMx only: the sheet of the annotation workbook to read (default: the first one with a Sample_ID column) |

In [ ]:
OPTIONS = {
    # One line per dataset (the folder's name under inputs/), only where you want to change something.
    # "my_folder":      {"kind": "xenium"},                # the table says "pick one", or got the kind wrong
    # "my_visium":      {"kind": "visium", "image": r"E:\scans\he_fullres.tif"},   # the image given to spaceranger
    # "nerve_section1": {"label": "Normal nerve"},
    # "neuroma_panel":  {"label": "Painful neuroma", "pixel_size": 0.5},
    # "big_scan":       {"src": r"E:\scans\slide_07.qptiff", "label": "Whole slide"},
    # "hd_section2":    {"user_layer": "008um"},   # + my_umap.csv / my_clusters.csv in its folder
}

import html, re
sys.path.insert(0, str(SV))
import detect            # web/SpatialViewer/detect.py: what `prepare.py detect` and `prepare.py auto` use
importlib.reload(detect)

# this notebook's names for three of prepare.py's kinds (also the folder names of the older inputs/<kind>/ layout)
NB_KIND = {"tiff": "proteomics_file", "akoya": "proteomics_markers", "rnaseq": "scrnaseq"}
GROUP = {"proteomics_file": "proteomics", "proteomics_markers": "proteomics", "visium": "transcriptomics",
         "xenium": "transcriptomics", "stereoseq": "transcriptomics", "cosmx": "transcriptomics", "merscope": "transcriptomics", "scrnaseq": "scrnaseq", "geomx": "transcriptomics"}
FILE_OPTIONS = ("image", "outs", "gef", "gem", "cellbin", "tissue_mask", "dcc", "pkc", "annotation", "cells_dir",
                "h5ad", "mtx", "umap", "obs", "images")    # OPTIONS keys that settle which files are read
HOW_SURE = {"certain": "certain", "ambiguous": "NOT SURE: pick one (see To do)",
            "incomplete": "recognised, but something is missing", "unknown": "not recognised"}

def safe_id(name):
    return re.sub(r"[^A-Za-z0-9_-]+", "_", str(name)).strip("_") or "dataset"

def user_files(ds_id, src, found, problems, notes):
    # Your own UMAP / clusters (Section 1): OPTIONS user_* keys, else my_umap.csv, my_clusters*.csv,
    # my_analysis.h5ad in the dataset folder
    opt = OPTIONS.get(ds_id, {})
    here = src if src.is_dir() else src.parent
    def path(p):
        p = Path(p)
        return p if p.is_absolute() else here / p
    umap = path(opt["user_umap"]) if "user_umap" in opt else here / "my_umap.csv"
    obs = opt.get("user_obs")
    obs = ([path(p) for p in ([obs] if isinstance(obs, (str, Path)) else obs)] if obs is not None
           else [here / "my_clusters.csv"] + sorted(here.glob("my_clusters_*.csv")))
    h5ad = path(opt["user_h5ad"]) if "user_h5ad" in opt else here / "my_analysis.h5ad"
    u = {"umap": umap if umap.exists() else None, "obs": [p for p in obs if p.exists()],
         "h5ad": h5ad if h5ad.exists() else None}
    for k, p in (("user_umap", umap), ("user_h5ad", h5ad)):
        if k in opt and not p.exists():
            problems.append(f"{k}: not found: {p}")
    if "user_obs" in opt:
        problems += [f"user_obs: not found: {p}" for p in obs if not p.exists()]
    if not (u["umap"] or u["obs"] or u["h5ad"]):
        return
    if u["h5ad"] and (u["umap"] or u["obs"]):
        problems.append("your own results: give either the .h5ad or the CSVs, not both")
    if not opt.get("user_layer"):
        problems.append("your own UMAP / clusters found: set user_layer (e.g. \"008um\") in OPTIONS")
    found["user"] = u
    names = [p.name for p in [u["umap"], u["h5ad"], *u["obs"]] if p]
    notes.append(f"your own results for layer {opt.get('user_layer')}: " + ", ".join(names))

def check_dataset(ds_id, src, hint=None, expand=True):
    # The rows for one folder (or file): detect.py's report, then this notebook's own checks.
    o = OPTIONS.get(ds_id, {})
    src = Path(o["src"]) if "src" in o else src
    try:
        rep = detect.detect(src, kind=o.get("kind"), hint=hint, overrides={k: o[k] for k in FILE_OPTIONS if k in o})
    except ValueError as e:                     # a "kind" that is not one of the kinds
        rep = detect.detect(src, hint=hint)
        rep.update(verdict="ambiguous", message=str(e))
    if rep["reason"] == "several-datasets" and expand:      # a folder of datasets: one row each, never merged
        subs = [Path(s["path"]) for s in rep["datasets"]]
        return [r for s in subs for r in check_dataset(safe_id(s.name if s.is_dir() else s.stem), s, hint, expand=False)]
    c = rep["candidates"][0] if rep["candidates"] else {"args": {}, "argv": [], "notes": [], "info": {}, "variant": ""}
    found = {k: [Path(x) for x in v] if isinstance(v, list) else Path(v) for k, v in c["args"].items()}   # the input files
    d = dict(id=ds_id, src=src, report=rep, kind=NB_KIND.get(rep["kind"], rep["kind"]) or "?", variant=c["variant"],
             found=found, argv=list(c["argv"]), problems=[], notes=list(c["notes"]) + list(rep["extras"]))
    if rep["verdict"] != "certain":
        d["problems"] += [rep["message"]] + detect.option_lines(rep, ds_id)
        return [d]
    if d["kind"] == "proteomics_markers" and "pixel_size" not in o:
        d["notes"].append("no pixel_size option: no scale bar or protein squares unless the files carry it")
    if d["kind"] == "visium":
        layers = c["info"].get("layers", [])
        lack = [b for b in VISIUM_BINS.split(",") if b not in layers] if (d["variant"] == "visium-hd" and VISIUM_BINS) else []
        if lack:
            d["problems"].append(f"VISIUM_BINS asks for {', '.join(lack)}, but outs/ has only {', '.join(layers)}")
        user_files(ds_id, Path(rep["datasets"][0]["path"]), d["found"], d["problems"], d["notes"])
    return [d]

def is_kind_folder(p):
    # inputs/visium/<id>/ ... of the older layout (not a dataset that happens to be called "visium")
    return p.name in GROUP and not any(s["root"] == "" for s in detect.detect(p)["datasets"])

def find_datasets():
    rows = []
    for p in sorted(INPUTS.iterdir(), key=lambda q: q.name.lower()):
        if p.name.startswith(".") or p.name == "README.txt":
            continue
        if p.is_dir() and is_kind_folder(p):
            for sub in sorted(q for q in p.iterdir() if q.is_dir()):
                rows += check_dataset(safe_id(sub.name), sub, hint=p.name)
        elif p.is_dir():
            rows += check_dataset(safe_id(p.name), p)
        else:                                    # a file on its own; stray files are not datasets
            rows += [r for r in check_dataset(safe_id(p.stem), p) if r["report"]["verdict"] != "unknown"]
    have = {r["id"] for r in rows}
    for ds_id, o in OPTIONS.items():             # datasets that only exist as a `src` somewhere else
        if "src" in o and ds_id not in have:
            rows += check_dataset(ds_id, Path(o["src"]), expand=False)
    seen = set()
    for r in rows:
        if r["id"] != safe_id(r["id"]):
            r["problems"].append("dataset id may only use letters, digits, _ and -")
        if r["id"] in seen:
            r["problems"].append(f"two datasets would get the id {r['id']}: rename one folder")
        seen.add(r["id"])
    return rows

def show(d):
    rep = d["report"]
    c = rep["candidates"][0] if rep["candidates"] else None
    found = [f"{e['what']}: {e['where']}" + (f" ({e['detail']})" if e["detail"] else "") for e in (c["evidence"] if c else [])]
    rows = [("Dataset id", d["id"]), ("Folder", str(d["src"])),
            ("Detected", rep["label"] or ", ".join(x["label"] for x in rep["candidates"]) or "-"),
            ("How sure", HOW_SURE[rep["verdict"]]), ("Found", found or ["-"]),
            ("To do", d["problems"] or ["nothing: ready"]), ("Notes", d["notes"] or ["-"])]
    if "get_ipython" in globals():
        from IPython.display import display, HTML
        cell = lambda v: "<br>".join(html.escape(str(x)) for x in (v if isinstance(v, list) else [v]))
        display(HTML("<table>" + "".join(
            f"<tr><th style='text-align:left;vertical-align:top;white-space:nowrap'>{k}</th>"
            f"<td style='text-align:left'>{cell(v)}</td></tr>" for k, v in rows) + "</table>"))
    else:
        for k, v in rows:
            for i, x in enumerate(v if isinstance(v, list) else [v]):
                print(f"  {k if i == 0 else '':11s} {x}")
        print()

DATASETS = find_datasets()
if not DATASETS:
    print(f"No datasets yet. Put one folder per dataset under {INPUTS} (Section 1), then run this cell again.")
for d in DATASETS:
    show(d)
READY = [d for d in DATASETS if not d["problems"]]
for d in DATASETS:
    print(f"{'READY' if not d['problems'] else 'FIX  '} {d['kind']:19s} {d['id']}")
print(f"\n{len(READY)} of {len(DATASETS)} dataset(s) ready." + ("" if len(READY) == len(DATASETS)
      else " For the others: see To do in their table, change the folder or OPTIONS, and run this cell again."))

Optional: look inside a proteomics or H&E file before building (axes, size, levels, channel
names). Nothing is written.

In [ ]:
for d in READY:
    if d["kind"] == "proteomics_file":
        subprocess.run([PY, str(SV / "prepare.py"), "inspect", "--src", str(d["found"]["src"])])

## 3. Prepare: read each dataset once into `cache/`

This is the slow step: it reads the full image once and builds a multi-resolution copy plus a
gene-by-bin index. The cache holds the full-resolution raw image, so **it stays on this
computer and is never uploaded**. A dataset with `cache/<id>/manifest.json` is skipped; set
`REBUILD = True` in the advanced settings (Section 0) to redo it (then also delete `export/<id>/`, or old tiles stay).

Single-cell RNA-seq has no cache: it is written straight to `export/` here.

In [ ]:
# Memory check before the slow step: total and free memory, the biggest dataset's input files,
# and the measured peaks from the README. It only warns; nothing is stopped.
import re

def memory_gb():
    # (total, free) memory in GB (2**30 bytes), or None when this system does not say
    try:
        import psutil
        vm = psutil.virtual_memory()
        return vm.total / 2**30, vm.available / 2**30
    except ImportError:
        pass
    try:
        if sys.platform == "win32":
            import ctypes
            class MemStatus(ctypes.Structure):
                _fields_ = [("dwLength", ctypes.c_ulong), ("dwMemoryLoad", ctypes.c_ulong)] + [
                    (n, ctypes.c_ulonglong) for n in ("ullTotalPhys", "ullAvailPhys", "ullTotalPageFile",
                                                      "ullAvailPageFile", "ullTotalVirtual", "ullAvailVirtual",
                                                      "ullAvailExtendedVirtual")]
            m = MemStatus()
            m.dwLength = ctypes.sizeof(MemStatus)
            if ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(m)):
                return m.ullTotalPhys / 2**30, m.ullAvailPhys / 2**30
        elif Path("/proc/meminfo").exists():                       # Linux
            kb = {}
            for line in open("/proc/meminfo"):
                parts = line.split()
                if len(parts) >= 2 and parts[1].isdigit():
                    kb[parts[0].rstrip(":")] = int(parts[1])
            return kb["MemTotal"] / 2**20, kb.get("MemAvailable", kb.get("MemFree", 0)) / 2**20
        elif sys.platform == "darwin":                             # macOS
            total = int(subprocess.run(["sysctl", "-n", "hw.memsize"], capture_output=True, text=True).stdout)
            vm = subprocess.run(["vm_stat"], capture_output=True, text=True).stdout
            page = int(re.search(r"page size of (\d+)", vm).group(1))
            pages = sum(int(n) for _, n in re.findall(r"Pages (free|inactive|speculative):\s+(\d+)", vm))
            return total / 2**30, pages * page / 2**30
    except Exception:
        pass
    return None

def files_size(p):
    p = Path(p)
    if p.is_file():
        return p.stat().st_size
    if p.is_dir():
        return sum(q.stat().st_size for q in p.rglob("*") if q.is_file())
    return 0

def input_gb(d):
    # About the bytes prepare.py reads for this dataset: for Visium only the image and the count
    # matrices it uses; for a folder input every file in it (an upper bound)
    f = d["found"]
    if d["kind"] == "visium" and "outs" in f:
        paths = [f.get("image")]
        binned = f["outs"] / "binned_outputs"
        if binned.is_dir():
            for layer in binned.glob("square_*um"):
                if not VISIUM_BINS or layer.name[len("square_"):] in VISIUM_BINS.split(","):
                    paths += [layer / "filtered_feature_bc_matrix.h5", layer / "spatial" / "tissue_positions.parquet"]
        else:
            paths += [f["outs"] / "filtered_feature_bc_matrix.h5"]
    else:
        paths = [p for k, v in f.items() if k != "user" for p in (v if isinstance(v, list) else [v])]
    return sum(files_size(p) for p in paths if isinstance(p, (str, Path))) / 2**30

try:
    mem = memory_gb()
    if mem:
        print(f"memory: {mem[0]:.1f} GB in total, {mem[1]:.1f} GB free now")
    else:
        print("memory: this system does not report it here; compare with the numbers below yourself")
    sizes = sorted(((input_gb(d), d) for d in READY), key=lambda t: t[0], reverse=True)
    if sizes:
        gb, big = sizes[0]
        print(f"largest input: {big['id']} ({big['kind']}), about {gb:.2f} GB of input files")
        # Measured peaks (README, Requirements): the demo 0.6 GB; a CosMx slide's counts 0.76 GB;
        # a whole mouse brain Stereo-seq chip 2.7 GB to prepare (3.9 GB from tissue.gem.gz) and
        # 8.5 GB to export, the largest measured. Whole-section data: have more free than 8.5 GB.
        whole = gb >= 1.0                       # 1 GB of input or more: treat as a whole section
        need = 8.5 if whole else 1.0
        print(f"suggested free memory: more than {need} GB "
              + ("(whole-section data: the largest measured peak, a whole Stereo-seq chip, was 8.5 GB)"
                 if whole else "(small data: the demo peaked at 0.6 GB)"))
        hd2 = [d["id"] for d in READY if d["kind"] == "visium" and "outs" in d["found"]
               and (d["found"]["outs"] / "binned_outputs" / "square_002um").is_dir()
               and (not VISIUM_BINS or "002um" in VISIUM_BINS.split(","))]
        if hd2:
            print("note: the 2 um Visium HD layer is on for " + ", ".join(hd2) + "; it has not been measured "
                  "and is expected to need the most memory. Take 002um out of VISIUM_BINS if it runs out.")
        if mem and mem[1] < need:
            print(f"\nWARNING: only {mem[1]:.1f} GB free, less than the {need} GB suggested. Prepare or export may "
                  "run out of memory (a MemoryError, or a very slow computer). Close other programs, use a "
                  "computer with more memory, or prepare the biggest dataset on its own. Nothing is stopped: "
                  "run the next cell when ready.")
        elif mem:
            print("ok: more free memory than suggested")
    else:
        print("no ready datasets (Section 2)")
except Exception as e:                                             # a check, never a blocker
    print("memory check skipped:", e)

In [ ]:
# REBUILD is in the advanced settings (Section 0).

def run(args, logfile=None):
    # Run one script, print its output as it comes, keep a log. Returns True on success.
    print("$", " ".join(str(a) for a in args))
    t0 = time.time()
    log = open(logfile, "a", encoding="utf-8") if logfile else None
    p = subprocess.Popen([str(a) for a in args], cwd=str(REPO), stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace",
                         env={**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8"})
    for line in p.stdout:
        print(line, end="")
        if log:
            log.write(line)
    p.wait()
    if log:
        log.close()
    print(f"-> exit {p.returncode} in {time.time() - t0:.0f} s\n")
    return p.returncode == 0

def prepare_args(d):
    # Section 2 found the files: d["argv"] is the kind and its file arguments. OPTIONS add the rest.
    o, k = OPTIONS.get(d["id"], {}), d["kind"]
    a = [PY, SV / "prepare.py"] + d["argv"] + ["--id", d["id"], "--out", CACHE, "--label", o.get("label", d["id"])]
    def opt(key, *kinds):
        v = o.get(key)
        if (not kinds or k in kinds) and v is not None and v is not False and v != "":
            a.extend(["--" + key.replace("_", "-")] + ([] if v is True else [v]))
    opt("pixel_size")
    opt("channels", "proteomics_file")
    if k == "visium" and d["variant"] == "visium-hd":
        a += ["--bins", VISIUM_BINS or "all"]      # "" = every layer (prepare.py alone keeps 8 and 16 um)
    opt("image_scale", "visium")
    for key in ("no_cells", "no_transcripts"):
        opt(key, "xenium", "cosmx", "merscope")
    opt("qv_min", "xenium")
    for key in ("images", "z"):
        opt(key, "cosmx", "merscope")
    opt("bins", "stereoseq")
    for key in ("sheet", "layout", "keep_slide_names"):
        opt(key, "geomx")
    for key in ("crop", "flip"):
        opt(key, "proteomics_file", "proteomics_markers", "visium")
    return a

def rnaseq_args(d):
    o = OPTIONS.get(d["id"], {})
    return [PY, SV / "prepare.py"] + d["argv"] + ["--id", d["id"], "--label", o.get("label", d["id"]), "--dest", EXPORT]

(WORK / "logs").mkdir(exist_ok=True)
results = {}
for d in READY:
    log = WORK / "logs" / f"{d['id']}_prepare.log"
    if d["kind"] == "scrnaseq":
        if (EXPORT / d["id"] / "manifest.json").exists() and not REBUILD:
            print(f"skip {d['id']}: already in export/"); results[d["id"]] = True; continue
        results[d["id"]] = run(rnaseq_args(d), log)
    else:
        if (CACHE / d["id"] / "manifest.json").exists() and not REBUILD:
            print(f"skip {d['id']}: already in cache/"); results[d["id"]] = True; continue
        results[d["id"]] = run(prepare_args(d), log)
print("prepared:", [k for k, v in results.items() if v])
print("FAILED:  ", [k for k, v in results.items() if not v] or "none")

## 4. Export: write the upload-ready files into `export/`

For every prepared image dataset this writes, into `export/<id>/`:

- `tiles/`: the image cut into 1024 x 1024 pieces at every zoom level, one channel per file;
- `expr/` (Visium): one file per gene per bin size, the bin positions, the whole-section UMAP
  with Leiden clusters (plus your own UMAP / clusters when you gave them, matched by barcode:
  the log says how many of your barcodes were found), and the data the in-browser circle -> UMAP
  tool reads;
- `expr/` (proteomics with a pixel size): the mean of each marker per 16 µm square, with a UMAP;
- `expr/rois/` (GeoMx): one file per target with the count of every ROI segment, the GeoMx QC
  values, and your annotation columns as groups (no UMAP for ROIs);
- `manifest.json`: the index the viewer reads (without your local file paths).

Then it writes **`export/datasets.json`**: the list of every dataset, the name the page shows
for it and the heading it sits under (`catalog.py`). Your link opens this list. You may rename
or regroup datasets in it with any text editor; running this cell again keeps your edits and
adds new datasets (see `docs/SHARE_YOUR_DATA.md`).

Re-running skips tiles and gene files that already exist. Check the log for
`[export] verified N channel(s)` and no line starting with `***`.

In [ ]:
exported = {}
for d in READY:
    if d["kind"] == "scrnaseq" or not results.get(d["id"]):
        continue
    a = [PY, SV / "prepare.py", "export", "--id", d["id"], "--out", CACHE, "--dest", EXPORT,
         "--fmt", TILE_FORMAT, "--workers", WORKERS]
    # gene data (Visium, Xenium, CosMx, MERSCOPE, Stereo-seq, GeoMx) takes the gene list,
    # protein images the protein list
    group = "proteomics" if d["kind"].startswith("proteomics") else "visium"
    if MARKERS.get(group):
        a += ["--markers", MARKERS[group]]
    u, opt = d["found"].get("user"), OPTIONS.get(d["id"], {})
    if u:                                        # your own UMAP / clusters (Section 1)
        a += ["--user-layer", opt["user_layer"]]
        a += ["--user-umap", u["umap"]] if u["umap"] else []
        for p in u["obs"]:
            a += ["--user-obs", p]
        a += ["--user-h5ad", u["h5ad"]] if u["h5ad"] else []
        for k in ("user_embedding", "user_columns", "user_barcode_prefix", "user_barcode_suffix"):
            if opt.get(k):
                a += ["--" + k.replace("_", "-"), opt[k]]
        a += ["--user-replace"] if opt.get("user_replace") else []
    exported[d["id"]] = run(a, WORK / "logs" / f"{d['id']}_export.log")
    txt = (WORK / "logs" / f"{d['id']}_export.log").read_text(encoding="utf-8", errors="replace")
    if "***" in txt:
        print(f"WARNING {d['id']}: the export log has a *** line; read it before uploading.")
print("exported:", [k for k, v in exported.items() if v])
print("FAILED:  ", [k for k, v in exported.items() if not v] or "none")

# The list of every dataset in export/ (datasets.json): the link you share opens this list.
if any((p / "manifest.json").exists() for p in EXPORT.iterdir() if p.is_dir()):
    print()
    if not run([PY, SV / "catalog.py", EXPORT] + (["--title", TITLE] if TITLE.strip() else [])):
        print("datasets.json was NOT written: read the [catalog] line above, fix it and run this cell again.")
else:
    print("No dataset in export/ yet, so no datasets.json either.")

## 5. De-identification check

Scans every file name and every text value in `export/` for anything that looks like a sample
ID. **It runs only when you have a key file** (`KEY_FILE` in the advanced settings): that file
means your lab de-identifies with `deid.py`. Without one it is skipped and says so; to run it
anyway, set `ALWAYS_CHECK_IDS = True` there.

When it runs, it must end with **`[deid] clean`**, and Section 7 will not continue until it
does. If it lists matches: rename the source (folder name, label, channel or column name),
delete `export/<id>/`, and run Sections 3-5 again. A gene name can match by accident, so read
the list before renaming anything.

Also read the labels and channel names yourself: the check only knows the ID patterns set in
`deid.py` (other labs: edit `_ID` there first, see `docs/DEIDENTIFICATION.md`).

In [ ]:
r = subprocess.run([PY, str(SV / "deid.py"), "check", str(EXPORT)], cwd=str(REPO),
                   capture_output=True, text=True, encoding="utf-8", errors="replace")
found_clean = r.returncode == 0 and "[deid] clean" in r.stdout
if KEY_FILE.exists() or ALWAYS_CHECK_IDS:
    # this lab de-identifies: the check is a gate
    print(r.stdout[-5000:] or r.stderr[-2000:])
    DEID_CLEAN = found_clean
    DEID_STATUS = "clean" if DEID_CLEAN else "not clean"
    print("\nOK to upload." if DEID_CLEAN else "\nNOT clean: fix the listed names before uploading. Section 7 stops until this says clean.")
else:
    # no key file: the check still runs, but only warns (its patterns are one lab's ID formats)
    DEID_CLEAN, DEID_STATUS = False, "skipped"
    if found_clean:
        print(f"Sample ID check: nothing found (no key file at {KEY_FILE}, so this is not a gate).")
    else:
        print(r.stdout[-5000:] or r.stderr[-2000:])
        print(f"\nWARNING: the names above look like sample IDs. No key file at {KEY_FILE}, so this does not stop the upload.\n"
              "If your lab de-identifies samples, set KEY_FILE in the advanced settings, run that cell and this one again,\n"
              "and fix the names before uploading. If these are not sample IDs, carry on.")


## 6. Look before you upload

The size of each dataset (many hosts limit file count and total size), then the exact
`export/` folder opened in your browser on this computer. The first link opens the whole
folder the way your shared link will. Nothing leaves your computer here.

In [ ]:
def folder_size(p):
    n = b = 0
    for f in p.rglob("*"):
        if f.is_file():
            n += 1; b += f.stat().st_size
    return n, b

tot_n = tot_b = 0
for ds in sorted(p for p in EXPORT.iterdir() if p.is_dir()):
    n, b = folder_size(ds)
    tot_n += n; tot_b += b
    print(f"{ds.name:40s} {n:>9,d} files {b / 1e9:8.2f} GB")
print(f"{'TOTAL':40s} {tot_n:>9,d} files {tot_b / 1e9:8.2f} GB")

In [ ]:
# Starts two small local servers: one for the export folder, one for the viewer page.
import socket
from urllib.parse import quote
def free_port():
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]
DATA_PORT, PAGE_PORT = free_port(), free_port()
servers = [
    subprocess.Popen([PY, str(SV / "static_serve.py"), str(EXPORT), str(DATA_PORT)], cwd=str(SV)),
    subprocess.Popen([PY, "-m", "http.server", str(PAGE_PORT), "--bind", "127.0.0.1", "--directory", str(SV)]),
]
time.sleep(1.5)
local_folder = f"http://127.0.0.1:{DATA_PORT}/"
print("All datasets, the way your shared link will open them:")
print(f"  http://127.0.0.1:{PAGE_PORT}/index.html?data={quote(local_folder, safe='')}")
if not (EXPORT / "datasets.json").exists():
    print("  (no datasets.json yet: run Section 4 to the end first)")
print("\nOne dataset at a time (also works when your config.js sets a password):")
for ds in sorted(p.name for p in EXPORT.iterdir() if p.is_dir()):
    print(f"  http://127.0.0.1:{PAGE_PORT}/index.html?static=http://127.0.0.1:{DATA_PORT}&ds={ds}")
print("\nOpen a link, click through every dataset, then run the next cell to stop the servers.")

In [ ]:
for s in servers:
    s.terminate()
print("servers stopped")

## 7. Upload and share

1. **Answer the question at the top of the next cell: is this data public?** The cell stops
   until you do.
2. Run the cell. It prints the upload command and **the link to share**. It runs nothing and
   uploads nothing.
3. Upload **the whole `export/` folder as it is** (one subfolder per dataset, plus
   `datasets.json`). Never upload `cache/`, `inputs/` or the key file.
4. Run the cell after it. It reads your uploaded folder the way a browser would and tells you
   whether the SODA page may read it. If not, it prints the exact setting to change.

What the answer means (see `docs/SHARE_YOUR_DATA.md` and `docs/HOSTING.md`):

- **`"yes"`, public**: anyone with the link can open (and download) the data. Only for data
  that may be published, such as examples and published data, with nothing about a patient.
  You pay for what people download, so keep public sets small.
- **`"no"`, private**: the data stays in a private folder. Readers either use your own
  password-protected SODA site (`deploy/azure/README.md`; the cell prints its `config.js`
  lines), or get a link that carries a read-only pass (`READ_PASS` in the advanced settings).
  Anyone holding such a link can read the data until the pass expires.

In [ ]:
# ANSWER THIS FIRST. Is this data public?
#   "yes" = anyone with the link may see and download it (published or example data, nothing about a patient)
#   "no"  = private: only people with your site's password or a read pass may see it
IS_THIS_DATA_PUBLIC = ""      # EDIT: "yes" or "no"

from urllib.parse import quote, urlsplit

answer = {True: "yes", False: "no"}.get(IS_THIS_DATA_PUBLIC, str(IS_THIS_DATA_PUBLIC).strip().lower())
if answer not in ("yes", "no"):
    raise ValueError('Answer the question first: set IS_THIS_DATA_PUBLIC = "yes" or "no" at the top of this cell.')
MODE = "public" if answer == "yes" else "private"

DEID_STATUS = globals().get("DEID_STATUS")
if DEID_STATUS is None:
    raise RuntimeError("Run Section 5 (the de-identification check) first.")
if DEID_STATUS == "not clean":
    raise RuntimeError("Section 5 found sample IDs. Fix them and run Section 5 again until it says clean.")
if DEID_STATUS == "skipped":
    print("Note: the sample ID check was skipped (no key file). Read the labels and channel names yourself.\n")
if not (EXPORT / "datasets.json").exists():
    print("Note: export/ has no datasets.json. Run Section 4 to the end before uploading.\n")

url = CLOUD_URL.strip()
if "?" in url:                              # a pasted pass (SAS) is a secret: drop it, never print it
    url = url.split("?", 1)[0]
    print("Note: removed the '?...' part of CLOUD_URL. Paste only the folder address there "
          "(a read-only pass goes in READ_PASS).\n")
base_url = url.rstrip("/")
if not base_url.startswith(("https://", "http://")) or "<" in base_url:
    raise ValueError("Set CLOUD_URL (Section 0) to your cloud folder's link, e.g. https://myaccount.blob.core.windows.net/spatial")
on_azure = ".blob.core.windows.net/" in base_url + "/"

read_pass = READ_PASS.strip().lstrip("?")
folder_link = base_url + "/" + ("?" + read_pass if read_pass else "")

def page_address(page):
    # the SODA page's index.html, or None when SODA_PAGE is not set
    p = (page or "").strip()
    if not p or "<" in p:
        return None
    if not p.startswith(("https://", "http://")):
        p = "https://" + p
    return p if p.split("?", 1)[0].endswith(".html") else p.rstrip("/") + "/index.html"

PAGE = page_address(SODA_PAGE)
SHARE_LINK = (PAGE or "<the SODA page's address>/index.html") + "?data=" + quote(folder_link, safe="")

lines = []
for ds in sorted(p for p in EXPORT.iterdir() if p.is_dir()):
    man = json.loads((ds / "manifest.json").read_text(encoding="utf-8"))
    d = next((x for x in DATASETS if x["id"] == ds.name), None)
    o = OPTIONS.get(ds.name, {})
    group = o.get("group") or (GROUP[d["kind"]] if d else "transcriptomics")
    label = o.get("label") or man.get("label") or ds.name
    entry = f'{ds.name}: {{ group: "{group}", label: {json.dumps(label)}'
    entry += f', base: "{base_url}/{ds.name}/" }},' if MODE == "public" else " },"
    lines.append("    " + entry)

print(f"This data is {MODE.upper()}.\n")
print("# 1. Upload export/ to", base_url)
if on_azure:
    print("# Make a short-lived WRITE pass (SAS) for the container first")
    print("# (Azure portal or Storage Explorer: container > Generate SAS, permissions Write + Create + List).")
    print('$sas = "<paste the SAS token here, starting with sv=>"')
    print(f'azcopy copy "{EXPORT}\\*" "{base_url}?$sas" --recursive')
else:
    print(f"# Copy everything inside {EXPORT} to that folder with your host's upload tool.")
if MODE == "public":
    print("# Public data: allow anonymous read access to the folder.")
print("# 2. Then run the next cell: it tests that the SODA page may read the folder (CORS).")

if MODE == "public" or read_pass:
    print("\n# 3. THE LINK TO SHARE:")
    print(SHARE_LINK)
    if not PAGE:
        print("# (SODA_PAGE is not set: replace <the SODA page's address> with the page's address,")
        print("#  or set SODA_PAGE in the advanced settings and run that cell and this one again.)")
    if read_pass:
        print("# This link carries a read pass: anyone holding it can read the data until the pass")
        print("# expires. Send it only to people who may see the data, like a password.")
else:
    print("\n# 3. No link to share: private data opens on your password-protected SODA site")
    print("#    (deploy/azure/README.md), with the config.js lines below. For a link instead,")
    print("#    set READ_PASS in the advanced settings to a READ-only pass and run this cell again.")

print("\n// Only for your own SODA site: paste into web/SpatialViewer/config.js (copy from config.example.js first)")
print("datasets: {\n" + "\n".join(lines) + "\n},")
print('tokenEndpoint: "/api/token",' if MODE == "private" else 'tokenEndpoint: "",')

### Test the link (after uploading)

A browser lets the SODA page read files from your storage only if the storage allows the
page's address (a CORS rule). This cell asks for your `datasets.json` the way a browser on
the SODA page would, then reads a few bytes of one gene pack with a Range request (that is how the
page reads one gene). If something is missing, it prints the exact fix for your storage.
Set it once, wait a minute, and run this cell again until it says OK.

In [ ]:
import urllib.request, urllib.error

def storage_kind(link):
    host = (urlsplit(link).hostname or "").lower()
    if host.endswith(".blob.core.windows.net"):
        return "azure"
    if host.endswith(".amazonaws.com") or host.endswith(".cloudfront.net"):
        return "s3"
    if host in ("storage.googleapis.com", "storage.cloud.google.com") or host.endswith(".storage.googleapis.com"):
        return "gcs"
    if host in ("localhost", "127.0.0.1", "::1"):
        return "local"
    return "web"

def cors_fix(link, origin):
    # the setting to change, per storage provider (as in docs/SHARE_YOUR_DATA.md, Step 4)
    o = origin or "<the SODA page's address, e.g. https://soda.example.org>"
    kind = storage_kind(link)
    if kind == "azure":
        return (f"Azure: Storage account -> Resource sharing (CORS) -> Blob service: Allowed origins {o},\n"
                "  Allowed methods GET, HEAD, OPTIONS, Allowed headers *, Exposed headers *, Max age 3600. Save.\n"
                "  (If you list headers instead of *: allow Range, expose Content-Range and Content-Length.)")
    if kind == "s3":
        return ("Amazon S3: Bucket -> Permissions -> Cross-origin resource sharing (CORS) -> Edit, paste:\n"
                f'  [{{ "AllowedOrigins": ["{o}"], "AllowedMethods": ["GET", "HEAD"],\n'
                '     "AllowedHeaders": ["*"], "ExposeHeaders": ["Content-Range", "Content-Length"], "MaxAgeSeconds": 3600 }]'
                + ("\n  (CloudFront: also forward the Origin header and the CORS response headers.)"
                   if (urlsplit(link).hostname or "").endswith(".cloudfront.net") else ""))
    if kind == "gcs":
        u = urlsplit(link)
        host = u.hostname or ""
        bucket = host[:-len(".storage.googleapis.com")] if host.endswith(".storage.googleapis.com") \
            else (u.path.strip("/").split("/") or ["<bucket>"])[0] or "<bucket>"
        return ("Google Cloud Storage: save this as cors.json\n"
                f'  [{{"origin": ["{o}"], "method": ["GET", "HEAD"], "responseHeader": ["Content-Type", "Range", "Content-Range", "Content-Length"], "maxAgeSeconds": 3600}}]\n'
                f"  and run: gcloud storage buckets update gs://{bucket} --cors-file=cors.json")
    if kind == "local":
        return ("A local test server: serve the folder with web/SpatialViewer/static_serve.py "
                "(it allows every page); python -m http.server does not.")
    return (f"A web server: send the header 'Access-Control-Allow-Origin: {o}' with every file of the folder,\n"
            "  'Access-Control-Expose-Headers: Content-Range, Content-Length', allow the Range header in a preflight,\n"
            "  and answer Range requests (the nginx example is in docs/HOSTING.md, 'Host notes').")

def range_test(root, qs, cat, origin, timeout=30):
    # The genes of an export are packed (a few large files per layer) and the page reads one gene
    # with an HTTP Range request. Ask for the first 12 bytes of one pack the way the page would:
    # the answer must be 206 with exactly those bytes, readable from the page's origin.
    def get(url, headers, method="GET"):
        req = urllib.request.Request(url, method=method, headers={"Origin": origin, "User-Agent": "SODA-notebook-link-test", **headers})
        try:
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return r.status, r.headers, r.read(1 << 16)
        except urllib.error.HTTPError as e:
            return e.code, e.headers, b""
    q = "?" + qs if qs else ""
    for d in cat.get("datasets", []):
        st, _, body = get(f"{root}/{d['id']}/manifest.json{q}", {})
        if st != 200:
            continue
        try:
            m = json.loads(body.decode("utf-8-sig"))
        except ValueError:
            continue
        packs = ([("", m["genePack"])] if m.get("genePack") else []) + \
                [(f"expr/{L['key']}/", L["genePack"]) for L in m.get("expr", []) if L.get("genePack")] + \
                ([(m["molecules"].get("dir", "molecules") + "/", m["molecules"]["genePack"])]
                 if (m.get("molecules") or {}).get("genePack") else [])
        if not packs:
            continue
        sub, pk = packs[0]
        path = f"{d['id']}/{sub}{pk.get('dir', 'genepack')}/{pk['shards'][0]['file']}"
        shown = f"{root}/{path}" + ("?<read pass>" if qs else "")
        st, h, body = get(f"{root}/{path}{q}", {"Range": "bytes=0-11"})
        if st == 200:
            return [f"{shown} came back whole (HTTP 200): this host ignores Range requests, so the page cannot read "
                    "one gene out of a pack. Use blob storage / S3 / a web server that answers Range, or export again "
                    "with --gene-files loose for this host."]
        if st != 206 or len(body) != 12:
            return [f"A Range request for {shown} answered HTTP {st} with {len(body)} bytes (want 206 and 12 bytes)."]
        expose = (h.get("Access-Control-Expose-Headers") or "").lower()
        if expose != "*" and "content-range" not in expose:
            return ["Range requests work, but the page cannot see Content-Range (CORS). Fix:\n  " + cors_fix(root, origin if PAGE else None)]
        st, h, _ = get(f"{root}/{path}{q}", {"Access-Control-Request-Method": "GET", "Access-Control-Request-Headers": "range"}, "OPTIONS")
        allow = (h.get("Access-Control-Allow-Headers") or "").lower()
        if st >= 400 or (allow != "*" and "range" not in allow):
            return [f"Browsers that ask before sending a Range header would be refused (CORS preflight: HTTP {st}, "
                    f"allowed headers {allow or 'none'}). Fix:\n  " + cors_fix(root, origin if PAGE else None)]
        return []
    return []                                  # nothing packed (exports with --gene-files loose): nothing to test


def cors_test(link, origin, timeout=30):
    # Fetch <folder>/datasets.json as a browser on `origin` would. Returns (ok, problems).
    root, _, qs = link.partition("?")
    url = root.rstrip("/") + "/datasets.json" + ("?" + qs if qs else "")
    shown = root.rstrip("/") + "/datasets.json" + ("?<read pass>" if qs else "")   # never print the pass
    req = urllib.request.Request(url, headers={"Origin": origin, "User-Agent": "SODA-notebook-link-test"})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            code, headers, body = r.status, r.headers, r.read()
    except urllib.error.HTTPError as e:
        code, headers, body = e.code, e.headers, b""
    except (urllib.error.URLError, OSError, ValueError) as e:
        return False, [f"Could not reach {shown}: {getattr(e, 'reason', e)}. Check CLOUD_URL."]
    problems = []
    if code == 404:
        problems.append(f"{shown} was not found (HTTP 404). Upload the whole export/ folder, with datasets.json, "
                        "to exactly this folder. (A private Azure container also answers 404 without a read pass.)")
    elif code in (401, 403, 409):
        problems.append(f"The storage refused {shown} (HTTP {code}): the folder is private. Public data: allow "
                        "anonymous read access. Private data: set READ_PASS to a read-only pass (or a new one if it expired).")
    elif code != 200:
        problems.append(f"{shown} answered HTTP {code}.")
    else:
        try:
            cat = json.loads(body.decode("utf-8-sig"))
            if not (isinstance(cat, dict) and isinstance(cat.get("datasets"), list) and cat["datasets"]):
                problems.append(f"{shown} lists no datasets. Run Section 4 to the end and upload datasets.json again.")
            elif (headers.get("Access-Control-Allow-Origin") or "").strip() in ("*", origin):
                problems += range_test(root.rstrip("/"), qs, cat, origin, timeout)
        except ValueError:
            problems.append(f"{shown} is not a SODA dataset list (is the link the export folder itself?).")
    allow = (headers.get("Access-Control-Allow-Origin") or "").strip()
    if allow not in ("*", origin):
        problems.append(f"The storage does not let the page on {origin} read it (CORS: "
                        + (f"it allows only {allow}" if allow else "no Access-Control-Allow-Origin header") + "). Fix:\n  "
                        + cors_fix(link, origin if PAGE else None))
    return not problems, problems

if MODE == "private" and not read_pass:
    print("Private data without a read pass: only your password-protected site can read the folder,")
    print("so this test cannot. Set the CORS rule for your site's address as in deploy/azure/README.md.")
else:
    origin = "{0.scheme}://{0.netloc}".format(urlsplit(PAGE)) if PAGE else "https://example.org"
    if not PAGE:
        print("SODA_PAGE is not set, so this tests as a page on https://example.org.")
        print("Set SODA_PAGE in the advanced settings to test with the real page's address.\n")
    ok, problems = cors_test(folder_link, origin)
    if ok:
        print(f"OK: a page on {origin} can read your folder.\n")
        print("THE LINK TO SHARE:")
        print(SHARE_LINK)
        if not PAGE:
            print("(replace <the SODA page's address> with the page's address first)")
        print("\nOpen it yourself first (a private browser window is best): you should see every dataset.")
    else:
        print("NOT READY yet:\n")
        for p in problems:
            print("- " + p + "\n")
        print("Change that, wait a minute, and run this cell again.")

### Before you share the link

- [ ] Section 5 says `[deid] clean` (or was skipped because you have no key file), and you read the labels and channel names yourself.
- [ ] You answered "is this data public?" with "yes" only for data anyone may see.
- [ ] Every dataset opened and looked right in Section 6.
- [ ] Only `export/` was uploaded: no `cache/`, raw files or key file.
- [ ] The link test says OK, and the link opened every dataset in a private browser window.
- [ ] The names in `datasets.json` (and `config.js`, if you have your own site) name the tissue and condition only.
- [ ] Private data: the password works and a link without it shows nothing; a link with a read pass goes only to people who may see the data.